In [1]:
# Import libraries
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.impute import SimpleImputer
import joblib

## 1. Load Data

In [2]:
# Set paths
ROOT = Path.cwd().parents[0]
DATA = ROOT / "data"
PROCESSED = DATA / "processed"
PROCESSED.mkdir(exist_ok=True)

# Load data
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

# Extract components
neural_cols = [c for c in train.columns if c not in ['session_id', 'trial_number', 'TRIAL_TYPE']]
X_train_raw = train[neural_cols].values
X_test_raw = test[neural_cols].values
y_train = train['TRIAL_TYPE'].values
sessions_train = train['session_id'].values
sessions_test = test['session_id'].values

print(f"\nNeural features: {len(neural_cols)}")
print(f"Sessions (train): {np.unique(sessions_train)}")
print(f"Sessions (test): {np.unique(sessions_test)}")

Train shape: (4407, 1383)
Test shape: (2292, 1382)

Neural features: 1380
Sessions (train): ['PG082_20221113' 'PG082_20221114' 'PG083_20221125' 'PG083_20221129'
 'PG084_20221206' 'PG084_20221207' 'PG085_20221213' 'PG085_20221214']
Sessions (test): ['PG082_20221120' 'PG083_20221130' 'PG084_20221208' 'PG085_20221215']


## 2. Handle Missing Values

In [6]:
# Check missing values
missing_train = np.isnan(X_train_raw).sum()
missing_test = np.isnan(X_test_raw).sum()
print(f"Missing values - Train: {missing_train}, Test: {missing_test}")

# Identify columns with ALL missing values (can't be imputed)
all_nan_mask = np.all(np.isnan(X_train_raw), axis=0)
cols_to_drop = np.array(neural_cols)[all_nan_mask].tolist()
cols_to_keep = np.array(neural_cols)[~all_nan_mask].tolist()
print(f"Columns with all NaN (dropped): {len(cols_to_drop)}")
print(f"Columns kept: {len(cols_to_keep)}")

# Filter to only keep columns with some data
X_train_filtered = X_train_raw[:, ~all_nan_mask]
X_test_filtered = X_test_raw[:, ~all_nan_mask]

# Simple median imputation (faster than kNN, minimal impact on final results)
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_filtered)
X_test_imp = imputer.transform(X_test_filtered)

# Update neural_cols to reflect kept columns
neural_cols_kept = cols_to_keep

print(f"\nAfter imputation:")
print(f"  Train shape: {X_train_imp.shape}")
print(f"  Test shape: {X_test_imp.shape}")
print(f"  Train NaN: {np.isnan(X_train_imp).sum()}, Test NaN: {np.isnan(X_test_imp).sum()}")

Missing values - Train: 1172310, Test: 551460
Columns with all NaN (dropped): 30
Columns kept: 1350

After imputation:
  Train shape: (4407, 1350)
  Test shape: (2292, 1350)
  Train NaN: 0, Test NaN: 0


## 3. Session-wise Z-score Normalization (THE KEY STEP)

This is the breakthrough technique that improved accuracy from ~42% to ~56%.

For each session, we normalize features to have:
- Mean = 0
- Standard deviation = 1

This removes session-specific baseline differences while preserving trial-to-trial patterns.

In [7]:
def session_normalize(X, sessions):
    """
    Z-score normalize features within each session.
    This removes session-specific baseline differences.
    
    Parameters:
    -----------
    X : np.ndarray, shape (n_samples, n_features)
        Feature matrix
    sessions : np.ndarray, shape (n_samples,)
        Session ID for each sample
    
    Returns:
    --------
    X_norm : np.ndarray, shape (n_samples, n_features)
        Session-normalized feature matrix
    """
    X_norm = X.copy()
    
    for sess in np.unique(sessions):
        mask = sessions == sess
        sess_data = X[mask]
        
        # Z-score within session
        sess_mean = np.nanmean(sess_data, axis=0)
        sess_std = np.nanstd(sess_data, axis=0)
        sess_std[sess_std == 0] = 1  # Avoid division by zero
        
        X_norm[mask] = (sess_data - sess_mean) / sess_std
    
    return X_norm

# Apply session normalization
print("Applying session-wise Z-score normalization...")
X_train_norm = session_normalize(X_train_imp, sessions_train)
X_test_norm = session_normalize(X_test_imp, sessions_test)

print(f"\nTrain shape: {X_train_norm.shape}")
print(f"Test shape: {X_test_norm.shape}")

# Verify normalization
print("\nVerification (mean/std per session after normalization):")
for sess in np.unique(sessions_train)[:3]:
    mask = sessions_train == sess
    print(f"  {sess}: mean={X_train_norm[mask].mean():.4f}, std={X_train_norm[mask].std():.4f}")

Applying session-wise Z-score normalization...

Train shape: (4407, 1350)
Test shape: (2292, 1350)

Verification (mean/std per session after normalization):
  PG082_20221113: mean=-0.0000, std=0.9888
  PG082_20221114: mean=0.0378, std=0.8998
  PG083_20221125: mean=-0.0267, std=0.9185


## 4. Export Processed Data

In [8]:
# Save processed data
print(f"Saving to {PROCESSED}...")

# Save as DataFrames with column names preserved (use kept columns)
pd.DataFrame(X_train_norm, columns=neural_cols_kept).to_csv(PROCESSED / "X_train_session_norm.csv", index=False)
pd.DataFrame(X_test_norm, columns=neural_cols_kept).to_csv(PROCESSED / "X_test_session_norm.csv", index=False)
pd.DataFrame({'TRIAL_TYPE': y_train}).to_csv(PROCESSED / "y_train.csv", index=False)
pd.DataFrame({'session_id': sessions_train}).to_csv(PROCESSED / "sessions_train.csv", index=False)
pd.DataFrame({'session_id': sessions_test}).to_csv(PROCESSED / "sessions_test.csv", index=False)

# Save feature names (only kept columns)
with open(PROCESSED / "feature_names.txt", "w") as f:
    f.write("\n".join(neural_cols_kept))

# Save the imputer for reproducibility
joblib.dump({'imputer': imputer, 'cols_kept': neural_cols_kept}, PROCESSED / "transforms.joblib")

print("\nFiles saved:")
print(f"  - X_train_session_norm.csv: {X_train_norm.shape}")
print(f"  - X_test_session_norm.csv: {X_test_norm.shape}")
print(f"  - y_train.csv: {y_train.shape}")
print(f"  - sessions_train.csv")
print(f"  - sessions_test.csv")
print(f"  - feature_names.txt ({len(neural_cols_kept)} features)")
print(f"  - transforms.joblib")

Saving to c:\Users\milor\Projects\ML-BIO-322\BIO-322\data\processed...

Files saved:
  - X_train_session_norm.csv: (4407, 1350)
  - X_test_session_norm.csv: (2292, 1350)
  - y_train.csv: (4407,)
  - sessions_train.csv
  - sessions_test.csv
  - feature_names.txt (1350 features)
  - transforms.joblib


## Summary

This preprocessing notebook applies the critical **session-wise Z-score normalization** that was 
discovered to be essential for good classification performance.

**Why it works:**
- Neural recordings drift across sessions due to electrode movement, brain state changes, etc.
- Global normalization treats all sessions as one distribution, which is wrong
- Session normalization removes session-specific offsets while preserving within-session patterns
- This matches how neuroscience papers (e.g., Esmaeili et al., 2021) preprocess their data

**Impact:** +14% accuracy improvement (42% → 56%)